In [17]:
from numba import njit, jit
from numpy import arange
import pandas as pd
import numpy as np
import threading
import multiprocessing

In [13]:
# jit decorator tells Numba to compile this function.
# The argument types will be inferred by Numba when function is called.

@jit
def sum2d(arr):
    M, N = arr.shape
    result = 0.0
    for i in range(M):
        for j in range(N):
            result += arr[i, j]
    return result

a = arange(9).reshape(3, 3)
print(f'Sum2D: \t\t{sum2d(a)}')
print(f'Numpy sum: \t{a.sum()}')

Sum2D: 		36.0
Numpy sum: 	36


In [14]:
x = {'a': [1, 2, 3], 'b': [20, 30, 40]}

def use_pandas(a):
    df = pd.DataFrame.from_dict(a)
    df += 1
    return df.cov()

@jit(forceobj=True) # Need to use object mode, try and compile loops!
def use_pandas_jit_obj(a): # Function will not benefit from Numba jit
    df = pd.DataFrame.from_dict(a) # Numba doesn't know about pd.DataFrame
    df += 1 # Numba doesn't understand what this is
    return df.cov() # or this!

use_pandas_jit_obj(x)

%timeit use_pandas(x)
%timeit use_pandas_jit_obj(x)

464 μs ± 10.1 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
634 μs ± 9.01 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [20]:
@jit(nopython=True, nogil=True)
def worker_nogil(arr1, arr2, arr3, chunk):
    """The thread worker."""
    for index in chunk:
        arr3[index] = arr1[index] + arr2[index]

def worker(arr1, arr2, arr3, chunk):
    """The thread worker."""
    for index in chunk:
        arr3[index] = arr1[index] + arr2[index]

nthreads = multiprocessing.cpu_count()
n = 1000000
a = np.random.randn(n)
b = np.random.randn(n)
c = np.empty(n, dtype='float64')

def run_with_n_threads(chunks, nthreads, worker):
    all_threads = []
    for chunk in chunks:
        thread = threading.Thread(target=worker, args=(a, b, c, chunk))
        all_threads.append(thread)
    for thread in all_threads:
        thread.start()
    for thread in all_threads:
        thread.join()

chunks = np.array_split(range(n), 1)
print("With GIL")
%timeit run_with_n_threads(chunks, 1, worker)
print("Without GIL")
%timeit run_with_n_threads(chunks, 1, worker_nogil)

chunks = np.array_split(range(n), 2)
print("With GIL")
%timeit run_with_n_threads(chunks,2, worker)
print("Without GIL")
%timeit run_with_n_threads(chunks,2, worker_nogil)

With GIL
523 ms ± 1.32 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
Without GIL
2.3 ms ± 12.6 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)
With GIL
526 ms ± 3.62 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
Without GIL
1.53 ms ± 33.4 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
